In [ ]:
# 1. Clone the repository
!git clone https://github.com/TheDatumOrg/TSB-AD
# STEP 2: Navigate into the cloned folder path (/content/TSB-AD)
%cd TSB-AD
# STEP 3: Install all required Python packages from the requirements text file
!pip install -r requirements.txt --verbose

In [ ]:
%cd TSB-AD
# STEP 4: Install the TSB-AD package in editable mode from the current directory
!pip install -e .
# STEP 5: Download the full zip file from the external URL address
!wget https://www.thedatum.org/datasets/TSB-AD-U.zip
!wget https://www.thedatum.org/datasets/TSB-AD-M.zip

In [ ]:

# 1. Unzip and automatically overwrite existing files (-o forces "yes")
!unzip -oq TSB-AD-U.zip -d Datasets/
!unzip -oq TSB-AD-M.zip -d Datasets/
# 2. Delete the zip files to free up disk space in Colab
!rm TSB-AD-U.zip TSB-AD-M.zip

In [ ]:
!python -m TSB_AD.main --AD_Name IForest

Performing anomaly detection using IFOREST approach. It uses NAB (Numenta Anomaly Benchmark) dataset, specifically the file 001_NAB_id_1_Facility_tr_1007_1st_2014.csv from the TSB-AD benchmark collection.

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# 1. Correct official imports from TSB-AD wrappers
from TSB_AD.model_wrapper import run_Unsupervise_AD
from TSB_AD.evaluation.metrics import get_metrics

# Dictionary to map metric names to descriptions and ideal target trends
METRIC_INFO = {
    'AUC-PR': ('Area Under Precision-Recall Curve (captures precision on imbalanced data)', 'Higher is better'),
    'AUC-ROC': ('Area Under Receiver Operating Characteristic Curve (overall separation capability)', 'Higher is better'),
    'VUS-PR': ('Volume Under the Surface for Precision-Recall (TSB-AD framework core range metric)', 'Higher is better'),
    'VUS-ROC': ('Volume Under the Surface for ROC (range-based structural spatial metric)', 'Higher is better'),
    'Standard-F1': ('Strict point-by-point harmonic mean of Precision and Recall', 'Higher is better'),
    'PA-F1': ('Point-Adjusted F1-score (forgives errors if a single anomaly point is hit)', 'Higher is better'),
    'Event-based-F1': ('F1 score calculated based on catching whole anomaly segments', 'Higher is better'),
    'R-based-F1': ('Range-based F1 score adjusting for overlap and fragmentation positional criteria', 'Higher is better'),
    'Affiliation-F': ('Precision/Recall score based on proximity intervals to true anomalies', 'Higher is better')
}

# 2. Define data path
data_path = 'Datasets/TSB-AD-U/001_NAB_id_1_Facility_tr_1007_1st_2014.csv'

if not os.path.exists(data_path):
    print(f"Error: Could not find the dataset at {data_path}")
    print("Please make sure your unzip command placed the files in the 'Datasets/' folder.")
else:
    # 3. Load the data using pandas
    df = pd.read_csv(data_path).dropna()

    # Extract continuous feature matrix and ground truth binary array
    data_features = df.iloc[:, 0:-1].values.astype(float)
    y_true = df['Label'].astype(int).to_numpy()

    # 4. Initialize framework anomaly detector execution wrapper
    print("Running Isolation Forest via TSB-AD framework wrapper...")
    AD_Name = 'IForest'
    y_score = run_Unsupervise_AD(AD_Name, data_features)

    # 5. Automatically calculate all benchmark metrics
    metrics_dict = get_metrics(y_score, y_true)

    # 6. Convert the metrics dictionary into a formatted pandas table with explanations
    table_data = []
    for metric_name, score_val in metrics_dict.items():
        desc, trend = METRIC_INFO.get(metric_name, ('Custom framework metric', 'Higher is better'))
        table_data.append({
            'Metric': metric_name,
            'Score': round(score_val, 4),
            'Definition': desc,
            'Ideal Trend': trend
        })

    df_metrics = pd.DataFrame(table_data).sort_values(by='Score', ascending=False)

    print("\n--- AUTOMATIC EVALUATION METRICS TABLE ---")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(df_metrics.to_string(index=False))
    print("\n" + "="*40 + "\n")

    # 7. Generate and display the horizontal bar graph
    plt.figure(figsize=(10, 5))
    bars = plt.barh(df_metrics['Metric'], df_metrics['Score'], color='#5c92e8', edgecolor='black', height=0.6)

    # Styling and adjustments
    plt.xlim(0, 1.0)
    plt.gca().invert_yaxis()  # Places the highest score at the top
    plt.xlabel('Score (0.0 to 1.0)', fontsize=11, fontweight='bold')
    plt.title(f'TSB-AD Metrics Graph (Algorithm: {AD_Name})', fontsize=13, fontweight='bold', pad=15)
    plt.grid(axis='x', linestyle='--', alpha=0.5)

    # Add numeric labels to the end of each bar
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.4f}',
                 va='center', ha='left', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

Detecting anomalies using an unsupervised PCA approach based on reconstruction error. The dataset used is NAB (Numenta Anomaly Benchmark) [001_NAB_id_1_Facility_tr_1007_1st_2014.csv]

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# 1. Correct official imports from TSB-AD wrappers
from TSB_AD.model_wrapper import run_Unsupervise_AD
from TSB_AD.evaluation.metrics import get_metrics

# Dictionary to map metric names to descriptions and ideal target trends
METRIC_INFO = {
    'AUC-PR': ('Area Under Precision-Recall Curve (captures precision on imbalanced data)', 'Higher is better'),
    'AUC-ROC': ('Area Under Receiver Operating Characteristic Curve (overall separation capability)', 'Higher is better'),
    'VUS-PR': ('Volume Under the Surface for Precision-Recall (TSB-AD framework core range metric)', 'Higher is better'),
    'VUS-ROC': ('Volume Under the Surface for ROC (range-based structural spatial metric)', 'Higher is better'),
    'Standard-F1': ('Strict point-by-point harmonic mean of Precision and Recall', 'Higher is better'),
    'PA-F1': ('Point-Adjusted F1-score (forgives errors if a single anomaly point is hit)', 'Higher is better'),
    'Event-based-F1': ('F1 score calculated based on catching whole anomaly segments', 'Higher is better'),
    'R-based-F1': ('Range-based F1 score adjusting for overlap and fragmentation positional criteria', 'Higher is better'),
    'Affiliation-F': ('Precision/Recall score based on proximity intervals to true anomalies', 'Higher is better')
}

# 2. Define data path
data_path = 'Datasets/TSB-AD-U/001_NAB_id_1_Facility_tr_1007_1st_2014.csv'

if not os.path.exists(data_path):
    print(f"Error: Could not find the dataset at {data_path}")
    print("Please make sure your unzip command placed the files in the 'Datasets/' folder.")
else:
    # 3. Load the data using pandas
    df = pd.read_csv(data_path).dropna()

    # Extract continuous feature matrix and ground truth binary array
    data_features = df.iloc[:, 0:-1].values.astype(float)
    y_true = df['Label'].astype(int).to_numpy()

    # 4. Initialize framework anomaly detector execution wrapper
    # CHANGED TO 'PCA'
    print("Running Principal Component Analysis (PCA) via TSB-AD framework wrapper...")
    AD_Name = 'PCA'
    y_score = run_Unsupervise_AD(AD_Name, data_features)

    # 5. Automatically calculate all benchmark metrics
    metrics_dict = get_metrics(y_score, y_true)

    # 6. Convert the metrics dictionary into a formatted pandas table with explanations
    table_data = []
    for metric_name, score_val in metrics_dict.items():
        desc, trend = METRIC_INFO.get(metric_name, ('Custom framework metric', 'Higher is better'))
        table_data.append({
            'Metric': metric_name,
            'Score': round(score_val, 4),
            'What it is': desc,
            'Ideal Trend': trend
        })

    df_metrics = pd.DataFrame(table_data).sort_values(by='Score', ascending=False)

    print("\n--- AUTOMATIC EVALUATION METRICS TABLE ---")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(df_metrics.to_string(index=False))
    print("\n" + "="*40 + "\n")

    # 7. Generate and display the horizontal bar graph
    plt.figure(figsize=(10, 5))
    bars = plt.barh(df_metrics['Metric'], df_metrics['Score'], color='#4caf50', edgecolor='black', height=0.6)

    # Styling and adjustments
    plt.xlim(0, 1.0)
    plt.gca().invert_yaxis()  # Places the highest score at the top
    plt.xlabel('Score (0.0 to 1.0)', fontsize=11, fontweight='bold')
    plt.title(f'TSB-AD Metrics Graph (Algorithm: {AD_Name})', fontsize=13, fontweight='bold', pad=15)
    plt.grid(axis='x', linestyle='--', alpha=0.5)

    # Add numeric labels to the end of each bar
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.4f}',
                 va='center', ha='left', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

Detecting anomalies using RPCA. dataset : NAB (Numenta Anomaly Benchmark) - 001_NAB_id_1_Facility_tr_1007_1st_2014.csv

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# 1. Correct official imports from TSB-AD wrappers
from TSB_AD.model_wrapper import run_Unsupervise_AD
from TSB_AD.evaluation.metrics import get_metrics

# Dictionary to map metric names to descriptions and ideal target trends
METRIC_INFO = {
    'AUC-PR': ('Area Under Precision-Recall Curve (captures precision on imbalanced data)', 'Higher is better'),
    'AUC-ROC': ('Area Under Receiver Operating Characteristic Curve (overall separation capability)', 'Higher is better'),
    'VUS-PR': ('Volume Under the Surface for Precision-Recall (TSB-AD framework core range metric)', 'Higher is better'),
    'VUS-ROC': ('Volume Under the Surface for ROC (range-based structural spatial metric)', 'Higher is better'),
    'Standard-F1': ('Strict point-by-point harmonic mean of Precision and Recall', 'Higher is better'),
    'PA-F1': ('Point-Adjusted F1-score (forgives errors if a single anomaly point is hit)', 'Higher is better'),
    'Event-based-F1': ('F1 score calculated based on catching whole anomaly segments', 'Higher is better'),
    'R-based-F1': ('Range-based F1 score adjusting for overlap and fragmentation positional criteria', 'Higher is better'),
    'Affiliation-F': ('Precision/Recall score based on proximity intervals to true anomalies', 'Higher is better')
}

# 2. Define data path
data_path = 'Datasets/TSB-AD-U/001_NAB_id_1_Facility_tr_1007_1st_2014.csv'

if not os.path.exists(data_path):
    print(f"Error: Could not find the dataset at {data_path}")
    print("Please make sure your unzip command placed the files in the 'Datasets/' folder.")
else:
    # 3. Load the data using pandas
    df = pd.read_csv(data_path).dropna()

    # Extract continuous feature matrix and ground truth binary array
    data_features = df.iloc[:, 0:-1].values.astype(float)
    y_true = df['Label'].astype(int).to_numpy()

    # 4. Initialize framework anomaly detector execution wrapper
    # CHANGED TO 'RobustPCA'
    print("Running Robust PCA (RobustPCA) via TSB-AD framework wrapper...")
    AD_Name = 'RobustPCA'
    y_score = run_Unsupervise_AD(AD_Name, data_features)

    # 5. Automatically calculate all benchmark metrics
    metrics_dict = get_metrics(y_score, y_true)

    # 6. Convert the metrics dictionary into a formatted pandas table with explanations
    table_data = []
    for metric_name, score_val in metrics_dict.items():
        desc, trend = METRIC_INFO.get(metric_name, ('Custom framework metric', 'Higher is better'))
        table_data.append({
            'Metric': metric_name,
            'Score': round(score_val, 4),
            'What it is': desc,
            'Ideal Trend': trend
        })

    df_metrics = pd.DataFrame(table_data).sort_values(by='Score', ascending=False)

    print("\n--- AUTOMATIC EVALUATION METRICS TABLE ---")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(df_metrics.to_string(index=False))
    print("\n" + "="*40 + "\n")

    # 7. Generate and display the horizontal bar graph
    plt.figure(figsize=(10, 5))
    bars = plt.barh(df_metrics['Metric'], df_metrics['Score'], color='#9c27b0', edgecolor='black', height=0.6)

    # Styling and adjustments
    plt.xlim(0, 1.0)
    plt.gca().invert_yaxis()  # Places the highest score at the top
    plt.xlabel('Score (0.0 to 1.0)', fontsize=11, fontweight='bold')
    plt.title(f'TSB-AD Metrics Graph (Algorithm: {AD_Name})', fontsize=13, fontweight='bold', pad=15)
    plt.grid(axis='x', linestyle='--', alpha=0.5)

    # Add numeric labels to the end of each bar
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.4f}',
                 va='center', ha='left', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

Comparing PCA and RPCA on the multivariate SMD (Server Machine Dataset) file 001_SMD_id_1.csv

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. Framework wrapper imports
from TSB_AD.model_wrapper import run_Unsupervise_AD
from TSB_AD.evaluation.metrics import get_metrics

# 2. Define your Multivariate file path
# Note: If your file has a slightly different name (like the tr_4529 one from your earlier logs),
# update this filename string to match an exact CSV inside your Datasets/TSB-AD-M/ folder.
data_path = 'Datasets/TSB-AD-M/001_SMD_id_1.csv'

# Fallback check: if the standard short name doesn't exist, we look for any CSV in that folder
if not os.path.exists(data_path) and os.path.exists('Datasets/TSB-AD-M/'):
    files = [f for f in os.listdir('Datasets/TSB-AD-M/') if f.endswith('.csv')]
    if files:
        data_path = os.path.join('Datasets/TSB-AD-M/', sorted(files)[0])

if not os.path.exists(data_path):
    print(f"Error: Could not find any multivariate dataset file.")
    print("Please verify that TSB-AD-M.zip was fully unzipped into 'Datasets/TSB-AD-M/'.")
else:
    print(f"Using Multivariate Dataset File: {data_path}\n")

    # 3. Load the multivariate data using pandas
    df = pd.read_csv(data_path).dropna()

    # Extract multiple continuous feature columns and the ground truth array
    # In multivariate files, all columns except 'Label' represent features
    data_features = df.drop(columns=['Label'], errors='ignore').values.astype(float)
    y_true = df['Label'].astype(int).to_numpy()

    print(f"Data shape loaded: {data_features.shape[0]} rows, {data_features.shape[1]} sensor columns.\n")

    # 4. Run Standard PCA
    print("1/2: Running Standard PCA...")
    try:
        y_score_pca = run_Unsupervise_AD('PCA', data_features)
        metrics_pca = get_metrics(y_score_pca, y_true)
    except Exception as e:
        print(f"PCA Execution failed: {e}")
        metrics_pca = {}

    # 5. Run Robust PCA
    print("2/2: Running Robust PCA...")
    try:
        y_score_rpca = run_Unsupervise_AD('RobustPCA', data_features)
        metrics_rpca = get_metrics(y_score_rpca, y_true)
    except Exception as e:
        print(f"RobustPCA Execution failed: {e}")
        metrics_rpca = {}

    # 6. Build a side-by-side comparison table
    all_metrics = sorted(list(set(list(metrics_pca.keys()) + list(metrics_rpca.keys()))))

    comparison_data = []
    for m in all_metrics:
        val_pca = metrics_pca.get(m, np.nan)
        val_rpca = metrics_rpca.get(m, np.nan)

        # Round numerical values for cleaner display
        score_pca = round(val_pca, 4) if not np.isnan(val_pca) else "N/A"
        score_rpca = round(val_rpca, 4) if not np.isnan(val_rpca) else "N/A"

        comparison_data.append({
            'Metric Target': m,
            'Standard PCA Score': score_pca,
            'Robust PCA Score': score_rpca
        })

    df_compare = pd.DataFrame(comparison_data)

    print("\n" + "="*50)
    print(" MULTIVARIATE ALGORITHM COMPARISON TABLE ")
    print("="*50)
    print(df_compare.to_string(index=False))
    print("="*50 + "\n")

    # 7. Generate a comparison chart
    # Filtering out metrics that returned NaN to keep the graph clean
    clean_metrics = [m for m in all_metrics if not np.isnan(metrics_pca.get(m, np.nan)) and not np.isnan(metrics_rpca.get(m, np.nan))]
    pca_vals = [metrics_pca[m] for m in clean_metrics]
    rpca_vals = [metrics_rpca[m] for m in clean_metrics]

    if clean_metrics:
        y_indices = np.arange(len(clean_metrics))
        bar_width = 0.35

        plt.figure(figsize=(12, 6))
        plt.barh(y_indices - bar_width/2, pca_vals, bar_width, label='Standard PCA', color='#4caf50', edgecolor='black')
        plt.barh(y_indices + bar_width/2, rpca_vals, bar_width, label='Robust PCA', color='#9c27b0', edgecolor='black')

        plt.xlim(0, 1.0)
        plt.yticks(y_indices, clean_metrics)
        plt.gca().invert_yaxis()  # Keep list ordered from top down
        plt.xlabel('Performance Score (0.0 to 1.0)', fontsize=11, fontweight='bold')
        plt.title('PCA vs Robust PCA Performance on Multivariate Data', fontsize=13, fontweight='bold', pad=15)
        plt.grid(axis='x', linestyle='--', alpha=0.5)
        plt.legend(loc='lower right')
        plt.tight_layout()
        plt.show()

In [ ]:
import os

folder = "Datasets/TSB-AD-M"
files = sorted(f for f in os.listdir(folder) if f.endswith(".csv"))

print(f"Found {len(files)} datasets")
print("\n".join(files[:30]))

Evaluating PCA & RobustPCA on Daphnet Human Activity Dataset. (018_Daphnet_id_1...csv)

In [ ]:
import pandas as pd
from TSB_AD.model_wrapper import run_Unsupervise_AD
from TSB_AD.evaluation.metrics import get_metrics

path = "Datasets/TSB-AD-M/018_Daphnet_id_1_HumanActivity_tr_9693_1st_20732.csv"
df = pd.read_csv(path).dropna()

X = df.drop(columns=["Label"], errors="ignore").to_numpy(dtype=float)
y = df["Label"].to_numpy(dtype=int)

print("Shape:", X.shape, "| Anomaly rate:", y.mean())

for name in ["PCA", "RobustPCA"]:
    scores = run_Unsupervise_AD(name, X)
    print(f"\n{name}:")
    print(get_metrics(scores, y))

Batch Benchmarking PCA vs RobustPCA Across 10 Multivariate Datasets

In [ ]:

import os, gc, io, contextlib, warnings
import numpy as np
import pandas as pd
from TSB_AD.model_wrapper import run_Unsupervise_AD
from TSB_AD.evaluation.metrics import get_metrics

folder = "Datasets/TSB-AD-M"
files = sorted(f for f in os.listdir(folder) if f.endswith(".csv"))[:10]
results = []

for i, filename in enumerate(files, 1):
    print(f"{i}/10: {filename}")
    row = {"Dataset": filename}

    try:
        df = pd.read_csv(os.path.join(folder, filename)).dropna()
        y = df["Label"].to_numpy(dtype=int)
        X = df.drop(columns=["Label"]).to_numpy(dtype=np.float64)

        if len(np.unique(y)) < 2 or not np.isfinite(X).all():
            row["Error"] = "Invalid data or only one label class"
        else:
            for name in ["PCA", "RobustPCA"]:
                try:
                    with warnings.catch_warnings(), \
                         contextlib.redirect_stdout(io.StringIO()):
                        warnings.simplefilter("ignore")
                        scores = run_Unsupervise_AD(name, X)
                        metrics = get_metrics(
                            np.asarray(scores).reshape(-1), y
                        )
                    for metric, value in metrics.items():
                        row[f"{name}_{metric}"] = float(value)
                    del scores
                except Exception as e:
                    row[f"{name}_Error"] = str(e)[:200]

        results.append(row)
        pd.DataFrame(results).to_csv(
            "/content/pca_robustpca_10_results.csv", index=False
        )

    except Exception as e:
        row["Error"] = str(e)[:200]
        results.append(row)

    finally:
        for var in ["df", "X", "y", "scores", "metrics"]:
            if var in globals():
                del globals()[var]
        gc.collect()

print("\nFinished. Results:")
print(pd.DataFrame(results).to_string(index=False))
print("\nSaved to /content/pca_robustpca_10_results.csv")


Standard PCA performs better on the first five datasets, showing higher precision and overall accuracy. However, Robust PCA performs significantly better on the last few sensor datasets (006, 009, and 010). This happens because Robust PCA filters out extreme noise and outliers that degrade standard PCA's performance, making it the superior choice for highly distorted multivariate data.

## PCA vs. Robust PCA: Results on Selected MSL Datasets

The following table compares PCA and Robust PCA across nine evaluation metrics on three MSL sensor datasets. **Bold values indicate the better score for each metric.**

| Metric | DB 006 PCA | DB 006 Robust PCA | DB 009 PCA | DB 009 Robust PCA | DB 010 PCA | DB 010 Robust PCA |
|---|---:|---:|---:|---:|---:|---:|
| AUC-PR | 0.056445 | **0.138105** | 0.004821 | **0.016405** | 0.025362 | **0.036054** |
| AUC-ROC | 0.253799 | **0.498571** | 0.211455 | **0.682061** | 0.417260 | **0.575955** |
| VUS-PR | 0.075483 | **0.166494** | 0.007197 | **0.039845** | 0.033765 | **0.059019** |
| VUS-ROC | 0.335555 | **0.589449** | 0.251809 | **0.864047** | 0.480365 | **0.677159** |
| Standard-F1 | 0.166076 | **0.287666** | 0.017744 | **0.059196** | **0.073915** | 0.073909 |
| PA-F1 | **0.367790** | 0.179618 | 0.014604 | **0.019654** | 0.125000 | **0.539062** |
| Event-based-F1 | **0.209607** | 0.153677 | 0.014604 | **0.019654** | 0.072271 | **0.074963** |
| R-based-F1 | **0.277294** | 0.249646 | 0.011245 | **0.013195** | **0.199747** | 0.084448 |
| Affiliation-F | **0.679686** | 0.671543 | **0.666623** | 0.666562 | **0.678476** | 0.675995 |

### Dataset Filenames

- **Database 006:** `006_MSL_id_5_Sensor_tr_1150_1st_1250.csv`
- **Database 009:** `009_MSL_id_8_Sensor_tr_714_1st_1390.csv`
- **Database 010:** `010_MSL_id_9_Sensor_tr_554_1st_1172.csv`

### Key Observation

Robust PCA achieves higher **VUS-PR** scores on all three datasets. Since VUS-PR is the primary metric of interest, these results suggest that Robust PCA may be more effective than standard PCA for anomaly detection on these particular datasets. However, the results vary across other metrics, so further investigation is needed to understand why.

Kernel PCA against PCA and Robust PCA using VUS-PR

In [ ]:

import os
import glob
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA

from TSB_AD.model_wrapper import run_Unsupervise_AD
from TSB_AD.evaluation.metrics import get_metrics

warnings.filterwarnings("ignore")

# Configuration
FOLDER = "Datasets/TSB-AD-M"
WINDOW = 100                  # Match repository PCA default
MAX_FIT_WINDOWS = 1500        # Limit kernel matrix memory
N_COMPONENTS = 20
BATCH_SIZE = 256
GAMMA_MULTIPLIERS = [0.1, 1.0, 10.0]

files = sorted(glob.glob(os.path.join(FOLDER, "*.csv")))[:10]
results = []


def vus_pr(scores, labels):
    scores = np.asarray(scores).ravel()
    labels = np.asarray(labels).ravel()

    if len(scores) != len(labels):
        raise ValueError(
            f"Score/label length mismatch: {len(scores)} vs {len(labels)}"
        )

    return float(get_metrics(scores, labels)["VUS-PR"])


def make_windows(X, window):
    # Each row contains the current observation and preceding observations.
    windows = np.lib.stride_tricks.sliding_window_view(
        X, window_shape=window, axis=0
    )
    return windows.transpose(0, 2, 1).reshape(len(X) - window + 1, -1)


def kpca_scores(X, window=100, max_fit=1500,
                n_components=20, gamma_multiplier=1.0):
    Xw = make_windows(X, window)

    # Deterministic, evenly spaced subset used to fit KPCA.
    n_fit = min(len(Xw), max_fit)
    fit_idx = np.linspace(
        0, len(Xw) - 1, n_fit, dtype=int
    )
    X_fit = Xw[fit_idx]

    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit)

    # Scale all windows in batches to control memory usage.
    n_features = X_fit_scaled.shape[1]
    gamma = gamma_multiplier / n_features

    model = KernelPCA(
        n_components=min(n_components, n_fit - 1),
        kernel="rbf",
        gamma=gamma,
        eigen_solver="arpack",
        fit_inverse_transform=True,
        alpha=1.0,
        random_state=0
    )
    model.fit(X_fit_scaled)

    # Reconstruction error in the original window feature space.
    scores = []
    for start in range(0, len(Xw), BATCH_SIZE):
        batch = scaler.transform(Xw[start:start + BATCH_SIZE])
        embedding = model.transform(batch)
        reconstructed = model.inverse_transform(embedding)

        error = np.mean((batch - reconstructed) ** 2, axis=1)
        scores.extend(error)

    scores = np.asarray(scores)

    # Align window scores to the original time-series length,
    # following the repository PCA's edge-padding convention.
    left_pad = int(np.ceil((window - 1) / 2))
    right_pad = int(np.floor((window - 1) / 2))

    scores = np.pad(scores, (left_pad, right_pad), mode="edge")
    return scores, gamma


for path in files:
    name = os.path.basename(path)
    print(f"\n{'=' * 65}\n{name}")

    df = pd.read_csv(path).dropna()
    label_col = next(
        (c for c in df.columns if c.lower() == "label"), None
    )

    if label_col is None:
        print("Skipped: label column not found.")
        continue

    y = (df[label_col].to_numpy() != 0).astype(int)

    X = df.drop(columns=[label_col]).to_numpy(dtype=float)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # Remove constant features.
    X = X[:, np.std(X, axis=0) > 1e-12]

    if len(X) <= WINDOW or X.shape[1] == 0:
        print("Skipped: insufficient rows or usable features.")
        continue

    if len(np.unique(y)) < 2:
        print("Skipped: labels contain only one class.")
        continue

    # Existing repository baselines.
    for algorithm in ["PCA", "RobustPCA"]:
        try:
            scores = run_Unsupervise_AD(algorithm, X)
            value = vus_pr(scores, y)

            results.append({
                "Dataset": name[:3],
                "Algorithm": algorithm,
                "Gamma": "-",
                "VUS-PR": value
            })
            print(f"{algorithm:12s} VUS-PR = {value:.5f}")

        except Exception as e:
            print(f"{algorithm} failed: {e}")

    # Kernel PCA with three RBF bandwidth settings.
    for multiplier in GAMMA_MULTIPLIERS:
        try:
            scores, gamma = kpca_scores(
                X,
                window=WINDOW,
                max_fit=MAX_FIT_WINDOWS,
                n_components=N_COMPONENTS,
                gamma_multiplier=multiplier
            )
            value = vus_pr(scores, y)

            results.append({
                "Dataset": name[:3],
                "Algorithm": "KernelPCA",
                "Gamma": f"{multiplier:g}/d",
                "VUS-PR": value
            })

            print(
                f"KernelPCA    gamma={multiplier:g}/d "
                f"(actual={gamma:.3g}) VUS-PR={value:.5f}"
            )

        except Exception as e:
            print(f"KernelPCA gamma={multiplier:g}/d failed: {e}")

    print("")

results_df = pd.DataFrame(results)

if not results_df.empty:
    print("\n========== ALL RESULTS ==========")
    print(
        results_df.pivot_table(
            index="Dataset",
            columns=["Algorithm", "Gamma"],
            values="VUS-PR"
        ).round(5).to_string()
    )

    print("\n========== BEST METHOD PER DATASET ==========")
    print(
        results_df.loc[
            results_df.groupby("Dataset")["VUS-PR"].idxmax()
        ].sort_values("Dataset").to_string(index=False)
    )


### Kernel PCA (RBF) vs. PCA and Robust PCA

Kernel PCA outperformed both PCA and Robust PCA on datasets 004 and 008, based on the VUS-PR metric.

| Dataset | PCA | Robust PCA | Kernel PCA | Best Method |
|---|---:|---:|---:|---|
| 004 | 0.76454 | 0.11553 | **0.79624** | Kernel PCA |
| 008 | 0.16392 | 0.12062 | **0.16777** | Kernel PCA |

**Explanation:** Kernel PCA achieved the highest VUS-PR score on both datasets, although the improvement over standard PCA was modest.

The results show that different algorithms win depending on the dataset structure:

- Standard PCA wins on 4 datasets (001, 002, 003, 005) because the data patterns are simple and linear.
- Robust PCA wins on 4 datasets (006, 007, 009, 010) because it successfully ignores extreme noise and outliers.
- Kernel PCA wins on 2 datasets (004, 008) because it captures complex, non-linear relationships that the other models miss entirely.

Changing the gamma hyperparameter has almost no effect on Kernel PCA's performance.

In [ ]:
# Kernel RPCA experiment on the first 10 datasets
# Compare VUS-PR against existing PCA and Robust PCA results.

!pip -q install scikit-learn

import os
import glob
import numpy as np
import pandas as pd
from sklearn.decomposition import KernelPCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

# Update this path if your dataset folder is elsewhere
DATASET_DIR = "Datasets/TSB-AD-M"

files = sorted(glob.glob(os.path.join(DATASET_DIR, "*.csv")))[:10]

print(f"Found {len(files)} datasets")
for f in files:
    print(os.path.basename(f))

Implementing and Testing Custom Kernel Robust PCA

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import StandardScaler
from TSB_AD.evaluation.metrics import get_metrics

def kernel_rpca_scores(X, gamma_scale=1.0, lam=None, max_iter=100, tol=1e-5):
    """
    Kernel robust decomposition using RBF Gram matrix
    and Principal Component Pursuit (low-rank + sparse).
    Returns an anomaly score for each time point.
    """
    X = np.asarray(X, dtype=float)

    if X.ndim == 1:
        X = X.reshape(-1, 1)

    X = StandardScaler().fit_transform(X)
    X = np.nan_to_num(X)

    n = len(X)
    if n < 3:
        return np.zeros(n)

    gamma = gamma_scale / max(X.shape[1], 1)
    K = rbf_kernel(X, gamma=gamma)
    K = (K + K.T) / 2

    lam = lam or 1.0 / np.sqrt(n)
    L = np.zeros_like(K)
    S = np.zeros_like(K)
    Y = np.zeros_like(K)

    # Inexact augmented Lagrangian method for low-rank + sparse
    norm2 = np.linalg.norm(K, 2)
    norm_inf = np.max(np.abs(K)) / lam
    dual_norm = max(norm2, norm_inf, 1e-12)
    Y = K / dual_norm

    mu = 1.25 / max(norm2, 1e-12)
    mu_max = mu * 1e7
    rho = 1.5

    for _ in range(max_iter):
        U, sigma, VT = np.linalg.svd(K - S + Y / mu, full_matrices=False)
        sigma_thresholded = np.maximum(sigma - 1 / mu, 0)
        L = (U * sigma_thresholded) @ VT

        residual = K - L + Y / mu
        S = np.sign(residual) * np.maximum(np.abs(residual) - lam / mu, 0)

        Z = K - L - S
        Y = Y + mu * Z
        mu = min(mu * rho, mu_max)

        if np.linalg.norm(Z, "fro") / (np.linalg.norm(K, "fro") + 1e-12) < tol:
            break

    # Per-point sparse-component magnitude as anomaly score
    scores = np.linalg.norm(S, axis=1)
    return scores


# Evaluate first 10 datasets using the same metric function
results_krpca = []

for file in files:
    df = pd.read_csv(file)

    # TSB-AD benchmark convention: final column contains labels
    data = df.to_numpy()
    X = data[:, :-1]
    labels = data[:, -1].astype(int)

    scores = kernel_rpca_scores(X)

    metrics = get_metrics(labels, scores)
    vus_pr = metrics["VUS-PR"]

    results_krpca.append({
        "Dataset": os.path.basename(file),
        "Kernel RPCA VUS-PR": vus_pr
    })

    print(f"{os.path.basename(file)}: VUS-PR = {vus_pr:.6f}")

krpca_df = pd.DataFrame(results_krpca)
display(krpca_df)

Fair Benchmarking of PCA vs. Kernel PCA Using Semi-Supervised Training

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
from sklearn.metrics.pairwise import euclidean_distances

from TSB_AD.evaluation.metrics import get_metrics

# =========================
# Configuration
# =========================
DATASET_DIR = "Datasets/TSB-AD-M"
N_DATASETS = 10
WINDOW_SIZE = 100
N_TRAIN_WINDOWS = 1500
N_COMPONENTS = 20
GAMMA_MULTIPLIER = 1.0
RANDOM_SEED = 42

# =========================
# Create sliding windows
# =========================
def make_windows(X, window_size):
    n = len(X)
    if n < window_size:
        return X.copy()

    return np.array([
        X[i:i + window_size].reshape(-1)
        for i in range(n - window_size + 1)
    ])

# =========================
# Load datasets
# =========================
files = sorted(glob.glob(os.path.join(DATASET_DIR, "*.csv")))[:N_DATASETS]

if not files:
    raise FileNotFoundError(f"No CSV files found in {DATASET_DIR}")

results = []

for dataset_idx, file_path in enumerate(files):
    print(f"\n{'='*60}")
    print(f"Dataset {dataset_idx + 1}/{len(files)}: {os.path.basename(file_path)}")

    df = pd.read_csv(file_path)

    if "Label" not in df.columns:
        print("Skipping: Label column not found.")
        continue

    labels = df["Label"].to_numpy().astype(int)

    X = df.drop(columns=["Label"]).select_dtypes(include=[np.number]).to_numpy()
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # Identical preprocessing for both methods
    X = StandardScaler().fit_transform(X)

    # Window vectors are used by both models
    X_windows = make_windows(X, WINDOW_SIZE)

    # Align each window with the label of its final time point
    if len(X_windows) == len(labels):
        eval_labels = labels
    else:
        eval_labels = labels[WINDOW_SIZE - 1:]

    n = min(len(X_windows), len(eval_labels))
    X_windows = X_windows[:n]
    eval_labels = eval_labels[:n]

    # Same normal-only training windows for both models
    normal_indices = np.flatnonzero(eval_labels == 0)

    if len(normal_indices) == 0:
        print("Skipping: no normal observations.")
        continue

    rng = np.random.default_rng(RANDOM_SEED)
    n_train = min(N_TRAIN_WINDOWS, len(normal_indices))
    train_indices = rng.choice(normal_indices, size=n_train, replace=False)

    X_train = X_windows[train_indices]

    # Standardize window features using training data only
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_eval = scaler.transform(X_windows)

    # Limit dimensionality if the training sample is too small
    n_components = min(N_COMPONENTS, X_train.shape[0] - 1, X_train.shape[1])

    # =========================
    # 1. Standard PCA
    # =========================
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pca.fit(X_train)

    X_reconstructed_pca = pca.inverse_transform(pca.transform(X_eval))
    scores_pca = np.mean((X_eval - X_reconstructed_pca) ** 2, axis=1)

    # =========================
    # 2. Kernel PCA (RBF)
    # =========================
    # Same training data and component count.
    # Gamma is set to 1 / number of input features.
    gamma = GAMMA_MULTIPLIER / X_train.shape[1]

    # Fit inverse mapping on the training data to reconstruct inputs.
    kpca = KernelPCA(
        n_components=n_components,
        kernel="rbf",
        gamma=gamma,
        fit_inverse_transform=True,
        alpha=1e-3,
        eigen_solver="arpack",
        random_state=RANDOM_SEED
    )

    kpca.fit(X_train)

    # Transform evaluation points and reconstruct them
    X_eval_embedding = kpca.transform(X_eval)
    X_reconstructed_kpca = kpca.inverse_transform(X_eval_embedding)
    scores_kpca = np.mean((X_eval - X_reconstructed_kpca) ** 2, axis=1)

    # =========================
    # Evaluate both identically
    # =========================
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        vus_pca = get_metrics(scores_pca, eval_labels)["VUS-PR"]
        vus_kpca = get_metrics(scores_kpca, eval_labels)["VUS-PR"]

    results.append({
        "Dataset": os.path.basename(file_path),
        "Train windows": n_train,
        "Evaluation points": len(eval_labels),
        "PCA VUS-PR": vus_pca,
        "Kernel PCA VUS-PR": vus_kpca,
        "Winner": (
            "Kernel PCA" if vus_kpca > vus_pca
            else "PCA" if vus_pca > vus_kpca
            else "Tie"
        )
    })

    print(f"PCA VUS-PR:        {vus_pca:.6f}")
    print(f"Kernel PCA VUS-PR: {vus_kpca:.6f}")

    # Release memory between datasets
    del X, X_windows, X_train, X_eval
    del pca, kpca, X_reconstructed_pca, X_reconstructed_kpca

# =========================
# Results
# =========================
results_df = pd.DataFrame(results)

print("\n\nFINAL RESULTS")
display(results_df)

if len(results_df):
    print("\nAverage PCA VUS-PR:",
          results_df["PCA VUS-PR"].mean())
    print("Average Kernel PCA VUS-PR:",
          results_df["Kernel PCA VUS-PR"].mean())
    print("Kernel PCA wins:",
          (results_df["Winner"] == "Kernel PCA").sum(),
          "out of", len(results_df))

results_df.to_csv("pca_vs_kernel_pca_fair_comparison.csv", index=False)
print("\nSaved: pca_vs_kernel_pca_fair_comparison.csv")

Under these fair testing conditions, Kernel PCA is the clear overall winner:

- Higher Accuracy: Kernel PCA achieves a much higher average score (0.257) than standard PCA (0.187).

- Massive Breakthroughs: Kernel PCA completely crushes standard PCA on Dataset 7 (jumping from 0.05 to 0.36) and Dataset 8 (jumping from 0.46 to a near-perfect 0.92) because it captures complex, non-linear sensor patterns.

- Close Losses: When standard PCA wins, it only wins by a tiny fraction, meaning Kernel PCA is highly competitive even on simple linear data.

This script runs an optimized, high-speed version of Kernel Robust PCA (Kernel RPCA) across your first 10 datasets. Because standard Kernel RPCA requires too much memory and crashes Google Colab, this code applies two smart mathematical shortcuts: Nyström approximation (which uses only 300 random data points instead of the entire dataset to map the complex kernel space) and Randomized SVD (which dramatically accelerates the matrix decomposition process). Combined with strict memory management using 32-bit floats, the script quickly isolates anomaly scores, calculates the VUS-PR metrics for each file, and saves the final benchmark results without running out of RAM.

In [ ]:

import os
import glob
import gc
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.utils.extmath import randomized_svd
from TSB_AD.evaluation.metrics import get_metrics

# =========================
# Fixed configuration
# =========================
DATASET_DIR = "Datasets/TSB-AD-M"
N_DATASETS = 10
N_LANDMARKS = 300
RANK = 20
MAX_ITER = 30
TOL = 1e-4
SEED = 42

# =========================
# Approximate Kernel RPCA
# =========================
def kernel_rpca(X):
    n, d = X.shape
    gamma = 1.0 / d
    m = min(N_LANDMARKS, n)

    rng = np.random.default_rng(SEED)
    idx = rng.choice(n, size=m, replace=False)
    landmarks = X[idx]

    # Nyström RBF kernel approximation
    C = rbf_kernel(X, landmarks, gamma=gamma).astype(np.float32)
    W = rbf_kernel(landmarks, gamma=gamma).astype(np.float64)
    W = (W + W.T) / 2

    eigvals, eigvecs = np.linalg.eigh(W)
    eigvals = np.maximum(eigvals, 1e-8)
    W_inv_sqrt = (
        eigvecs * (1.0 / np.sqrt(eigvals))
    ) @ eigvecs.T

    Z = (C @ W_inv_sqrt.astype(np.float32)).astype(np.float32)

    del C, W, eigvals, eigvecs, W_inv_sqrt, landmarks
    gc.collect()

    # Robust low-rank + sparse decomposition: Z = L + S
    lam = 1.0 / np.sqrt(max(Z.shape))
    L = np.zeros_like(Z)
    S = np.zeros_like(Z)
    Y = np.zeros_like(Z)

    norm_Z = np.linalg.norm(Z, "fro") + 1e-12
    mu = 1.25 / (np.linalg.norm(Z, 2) + 1e-12)
    mu_max = mu * 1e7

    for it in range(MAX_ITER):
        A = Z - S + Y / mu

        k = min(RANK, min(A.shape) - 1)
        k = max(k, 1)

        U, s, Vt = randomized_svd(
            A,
            n_components=k,
            n_iter=2,
            random_state=SEED + it
        )

        s = np.maximum(s - 1.0 / mu, 0.0)
        L = ((U * s) @ Vt).astype(np.float32)

        R = Z - L + Y / mu
        S = (
            np.sign(R) * np.maximum(np.abs(R) - lam / mu, 0.0)
        ).astype(np.float32)

        error = Z - L - S
        rel_error = np.linalg.norm(error, "fro") / norm_Z
        Y += mu * error

        if it == 0 or (it + 1) % 5 == 0:
            print(
                f"  Iteration {it+1}/{MAX_ITER}: "
                f"error={rel_error:.6f}"
            )

        if rel_error < TOL:
            print(f"  Converged at iteration {it+1}")
            break

        mu = min(mu * 1.5, mu_max)

        del A, U, s, Vt, R, error

    scores = np.linalg.norm(S, axis=1)

    del Z, L, S, Y
    gc.collect()
    return scores


# =========================
# Evaluate first 10 datasets
# =========================
files = sorted(
    glob.glob(os.path.join(DATASET_DIR, "*.csv"))
)[:N_DATASETS]

if not files:
    raise FileNotFoundError(f"No CSV files found in {DATASET_DIR}")

results = []

for i, path in enumerate(files, start=1):
    print("\n" + "=" * 60)
    print(f"Dataset {i}/{len(files)}: {os.path.basename(path)}")

    df = pd.read_csv(path)

    if "Label" not in df.columns:
        print("Skipped: Label column missing.")
        continue

    labels = df["Label"].to_numpy().astype(int)

    X = df.drop(columns=["Label"]).select_dtypes(
        include=[np.number]
    ).to_numpy()

    X = np.nan_to_num(
        X, nan=0.0, posinf=0.0, neginf=0.0
    )

    print("Points:", len(X))
    print("Features:", X.shape[1])
    print("Anomalies:", int(labels.sum()))

    # Consistent per-dataset feature standardization.
    # This uses all unlabeled observations, so treat results
    # as an offline/transductive benchmark.
    X = StandardScaler().fit_transform(X).astype(np.float32)

    try:
        scores = kernel_rpca(X)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            vus_pr = get_metrics(scores, labels)["VUS-PR"]

        print(f"Kernel RPCA VUS-PR: {vus_pr:.6f}")

        results.append({
            "Dataset": os.path.basename(path),
            "Points": len(X),
            "Anomalies": int(labels.sum()),
            "Landmarks": min(N_LANDMARKS, len(X)),
            "Rank": RANK,
            "VUS-PR": float(vus_pr)
        })

    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")
        results.append({
            "Dataset": os.path.basename(path),
            "Points": len(X),
            "Anomalies": int(labels.sum()),
            "Landmarks": min(N_LANDMARKS, len(X)),
            "Rank": RANK,
            "VUS-PR": np.nan
        })

    del df, X, labels
    if "scores" in locals():
        del scores
    gc.collect()

# =========================
# Final results
# =========================
results_df = pd.DataFrame(results)

print("\nFINAL RESULTS")
display(results_df)

results_df.to_csv(
    "kernel_rpca_nystrom_first10.csv",
    index=False
)
print("Saved: kernel_rpca_nystrom_first10.csv")


The result shows that the approximate Kernel RPCA algorithm runs fast, converges perfectly, but provides lower overall accuracy compared to the full window-based models:

- Fast Convergence: The optimization loop works flawlessly, with every single dataset successfully converging in fewer than 30 iterations (usually between 23 and 28 loops).

- Performance Drop: The average accuracy is noticeably lower because it lacks the sliding window context. For instance, on Dataset 8, this fast approximation drops to a 0.221 VUS-PR score, whereas the previous full-window Kernel PCA reached a near-perfect 0.921.

- The Trade-off: While the Nyström approximation effectively solves the RAM crash problem and handles large point clouds (like the 16,220 points in Dataset 1), compressing the kernel mapping to just 300 landmark points causes the model to miss crucial sequential, time-series anomaly patterns.


In the paper "The Elephant in the Room: Towards A Reliable Time-Series Anomaly Detection Benchmark (TSB-AD)" Standard PCA coross their evaluations:

- Univariate Benchmark (TSB-AD-U): Sub-PCA ranks 1st overall out of 32 algorithms under the reliable VUS-PR metric, whereas Robust PCA fails to make the top 12.

- Multivariate Benchmark (TSB-AD-M): Standard PCA ranks 3rd overall (VUS-PR of 0.31), while Robust PCA ranks 12th (VUS-PR of 0.24).

Why Robust PCA Underperforms on Time-Series Data

• Fails on Sequence Anomalies: RPCA assumes anomalies are short, isolated spikes. Long, continuous time-series anomaly blocks leak directly into the normal data matrix.

• Hyperparameter Sensitivity: It relies heavily on an exact tuning parameter LAMBDA. Slight errors push normal high-variance trends into the outlier matrix, creating massive false positives.

• Volatile Scoring: Its harsh splitting creates jumpy anomaly scores. Standard PCA's smooth reconstruction error matches boundary-tolerant metrics like VUS-PR much better.

The Elephant in the Room (TSB-AD) paper assesses a total of 40 anomaly detection algorithms. They evaluate these models across three broad architectural categories:


1. Statistical Methods (25 Algorithms)

This category includes classical distance, density, projection, and cluster-based baselines. Key models evaluated are:

* Projection / Subspace: Standard PCA (Sub-PCA), Robust PCA (RobustPCA), and Extended Isolation Forest (EIF).

* Proximity / Density: Isolation Forest (IForest), Local Outlier Factor (LOF), K-Nearest Neighbors (KNN), and Minimum Covariance Determinant (MCD).

* Clustering: KMeansAD, CBLOF, KShapeAD, and SAND.

* Time-Series / Other: MatrixProfile, Series2Graph, Spectral Residual (SR), POLY, and COPOD.


2. Neural Network Methods (10 Algorithms)

This category comprises deep learning architectures optimized for sequential reconstruction or prediction:

* Reconstruction / Generative: AutoEncoder, Variational AutoEncoder (Donut), OmniAnomaly, and USAD.

* Temporal / Attention: LSTMAD, CNN, AnomalyTransformer, and TranAD.

* General / Scaled: TimesNet and FITS.


3. Foundation Models (5 Algorithms)

This represents the latest generation of time-series foundation models evaluated in both zero-shot (ZS) and fine-tuned (FT) settings:

* MOMENT (A masked time-series encoder backbone).
* TimesFM (Decoder-style attention model).
* Chronos (Quantized T5 language model adaptation).
* Lag-Llama (Probabilistic forecasting transformer).
* OFA (One Fits All - fine-tuned GPT-2 backbone).